<a href="https://colab.research.google.com/github/skyline180/ChitroJera-Multimodal-VQA/blob/main/ChitroJera_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environment Setup & Installations

In [ ]:
# Install the necessary open-source AI and data libraries
!pip install -q transformers accelerate datasets evaluation tqdm
!pip install -q opendatasets

In [ ]:

import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import torchvision.transforms as transforms
from transformers import AutoTokenizer, ViTModel, AutoModel
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
# Assign the default hardware acceleration engine
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using target device: {device}")

# Dataset Loading and Preprocessing

In [ ]:
import opendatasets as od

# Download ChitroJera from Kaggle to Colab's local drive storage filesystem
# This prompt will request your Kaggle Username and Secret API Token
try:
    od.download("https://www.kaggle.com/datasets/sourove/chitrojera")
    print("Download request processed successfully!")
except Exception as e:
    print(f"Dataset handler exception captured: {e}")

# Base virtual environment directories mapping
DATASET_PATH = "/content/chitrojera"

# Automatically resolve the correct annotation file naming variant inside the download folder
csv_candidate_1 = os.path.join(DATASET_PATH, "chitrojera.csv")
csv_candidate_2 = os.path.join(DATASET_PATH, "metadata.csv")

if os.path.exists(csv_candidate_1):
    ANNOTATIONS_CSV = csv_candidate_1
elif os.path.exists(csv_candidate_2):
    ANNOTATIONS_CSV = csv_candidate_2
else:
    ANNOTATIONS_CSV = None

# Fallback structure logic to allow notebook execution verification if files aren't downloaded
if ANNOTATIONS_CSV and os.path.exists(ANNOTATIONS_CSV):
    df = pd.read_csv(ANNOTATIONS_CSV)
    print("=" * 60)
    print(f"✅ SUCCESS: Connected to the REAL 15k+ ChitroJera Benchmark Dataset!")
    print(f"📊 Total Loaded Workspace Entries: {len(df)} rows")
    print("=" * 60)
    df['question'] = df['question'].astype(str)
    df['answer'] = df['answer'].astype(str)
else:
    print("⚠️ Real CSV path not detected yet. Building local sample simulation rows...")
    mock_data = {
        'image_path': ['sample_img1.jpg', 'sample_img2.jpg'],
        'question': ['এই ছবিতে কী দেখা যাচ্ছে?', 'ক্রিকেটার কোন জার্সি পরে আছেন?'],
        'answer': ['ক্রিকেট ম্যাচ', 'লাল-সবুজ']
    }
    df = pd.DataFrame(mock_data)
    Image.new('RGB', (224, 224), color='red').save('sample_img1.jpg')
    Image.new('RGB', (224, 224), color='green').save('sample_img2.jpg')
    print(f"📊 Simulation Dataset Size: {len(df)} rows")

print(df.head(2))

# PyTorch Custom Dataset Pipeline

In [ ]:
class ChitroJeraDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

        # Generate classification vocabulary indices from target responses
        self.ans_to_idx = {ans: i for i, ans in enumerate(self.df['answer'].unique())}
        self.idx_to_ans = {i: ans for ans, i in self.ans_to_idx.items()}

        # Vision Transformer preprocessing transformations
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_path']

        # Build absolute directory pathing string if using Kaggle dataset paths
        if not os.path.isabs(img_path) and os.path.exists(os.path.join(DATASET_PATH, img_path)):
            img_path = os.path.join(DATASET_PATH, img_path)

        try:
            image = Image.open(img_path).convert('RGB')
            image_tensor = self.transform(image)
        except Exception:
            # Safeguard error fallback using clear zeros baseline tensor matrix
            image_tensor = torch.zeros(3, 224, 224)

        # Tokenize Bengali context question string
        question_text = str(row['question'])
        inputs = self.tokenizer(
            question_text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        label = self.ans_to_idx.get(row['answer'], 0)

        return {
            'image': image_tensor,
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Defining the ChitroJera Dual-Encoder Framework

In [ ]:
class ChitroJeraDualEncoderVQA(nn.Module):
    def __init__(self, text_model_name="csebuetnlp/banglabert", num_classes=2):
        super(ChitroJeraDualEncoderVQA, self).__init__()

        # Text Encoder Component (BanglaBERT)
        self.text_encoder = AutoModel.from_pretrained(text_model_name)

        # Vision Encoder Component (ViT Base)
        self.vision_encoder = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")

        # Linear projection vectors to map embeddings to identical feature dimensions
        self.text_projection = nn.Linear(self.text_encoder.config.hidden_size, 512)
        self.vision_projection = nn.Linear(self.vision_encoder.config.hidden_size, 512)

        # Deep multimodal classification block
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask, images):
        # Extract features from textual inputs via CLS pooling token
        text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_cls = text_outputs.last_hidden_state[:, 0, :]
        text_features = self.text_projection(text_cls)

        # Extract features from visual images via vision transformer pooling layer
        vision_outputs = self.vision_encoder(pixel_values=images)
        vision_cls = vision_outputs.last_hidden_state[:, 0, :]
        vision_features = self.vision_projection(vision_cls)

        # Late Fusion cross-modal concatenating step
        fused_features = torch.cat((text_features, vision_features), dim=-1)

        # Compute final classification logits output
        logits = self.classifier(fused_features)
        return logits

# Training and Evaluation Phase Configuration Loops

In [ ]:
# Configuration Parameters Hyper-parameters Block
BATCH_SIZE = 16
EPOCHS = 10             # Scale this up to 15-20 later for deeper model convergence
LEARNING_RATE = 2e-5

print("Preparing Dataset splits and DataLoaders...")
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/banglabert")

train_dataset = ChitroJeraDataset(train_df, tokenizer)
val_dataset = ChitroJeraDataset(val_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

# Initialization
num_labels = len(train_dataset.ans_to_idx)
model = ChitroJeraDualEncoderVQA(num_classes=num_labels).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

best_val_loss = float('inf')

print(f"Beginning training loop optimization routine on target hardware device: {device}...\n")

for epoch in range(EPOCHS):
    # --- TRAINING WORKFLOW BLOCK ---
    model.train()
    total_train_loss = 0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Training]")

    for batch in train_loop:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        images = batch['image'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids, attention_mask, images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        train_loop.set_postfix(loss=loss.item())

    avg_train_loss = total_train_loss / len(train_loader)

    # --- VALIDATION EVALUATION BLOCK ---
    model.eval()
    total_val_loss = 0
    correct_predictions = 0
    total_predictions = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            images = batch['image'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask, images)
            loss = criterion(outputs, labels)
            total_val_loss += loss.item()

            predictions = torch.argmax(outputs, dim=-1)
            correct_predictions += (predictions == labels).sum().item()
            total_predictions += labels.size(0)

    avg_val_loss = total_val_loss / len(val_loader)
    val_accuracy = (correct_predictions / total_predictions) * 100

    print(f"\n📈 [Epoch {epoch+1} Complete Evaluation Metrics]:")
    print(f"   ↳ Mean Training Loss Vector: {avg_train_loss:.4f}")
    print(f"   ↳ Mean Validation Loss Vector: {avg_val_loss:.4f}")
    print(f"   ↳ Measured Validation Accuracy: {val_accuracy:.2f}%\n")

    # Save checkpoint parameter files conditionally based on metrics improvements
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "chitrojera_dual_encoder.pt")
        print("💾 Loss convergence verified! Saving checkpoint weights tracking parameters to file.\n")
    else:
        print("⚠️ Validation performance metrics did not exceed previous thresholds. Skipping weights storage save.\n")

print("🏁 End-to-end model training execution sequence terminated.")

# Independent Inference Testing Block on Unseen Test Split

In [ ]:
# Reconstruct clean framework structure to evaluate inference from the saved checkpoint file
inference_model = ChitroJeraDualEncoderVQA(num_classes=num_labels).to(device)
inference_model.load_state_dict(torch.load("chitrojera_dual_encoder.pt", map_location=device))
inference_model.eval()

def ask_chitrojera(image_path, question_text):
    image = Image.open(image_path).convert('RGB')
    image_tensor = val_dataset.transform(image).unsqueeze(0).to(device)

    inputs = tokenizer(question_text, padding='max_length', truncation=True, max_length=128, return_tensors="pt")
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)

    with torch.no_grad():
        logits = inference_model(input_ids, attention_mask, image_tensor)
        predicted_idx = torch.argmax(logits, dim=-1).item()

    return val_dataset.idx_to_ans.get(predicted_idx, "Unknown")

# Pick an arbitrary sample row from the validation split (val_df)
random_row = val_df.sample(n=1).iloc[0]
raw_image_path = random_row['image_path']

# Resolve exact file position on server
if not os.path.isabs(raw_image_path) and os.path.exists(os.path.join(DATASET_PATH, raw_image_path)):
    absolute_path = os.path.join(DATASET_PATH, raw_image_path)
else:
    absolute_path = raw_image_path

print("=" * 60)
print(f"📍 Testing Image Path: {absolute_path}")
print(f"❓ Text Question Input: {random_row['question']}")
print(f"🎯 Target Ground Truth: {random_row['answer']}")

if os.path.exists(absolute_path):
    model_prediction = ask_chitrojera(absolute_path, random_row['question'])
    print(f"🤖 Dual-Encoder AI Prediction: {model_prediction}")
    print("=" * 60)

    # Display the visual content file inside your notebook interface
    plt.figure(figsize=(5, 5))
    plt.imshow(Image.open(absolute_path))
    plt.axis('off')
    plt.show()
else:
    print("❌ Testing asset file path unresolvable.")

In [ ]:
# end